In [ ]:
dataset_url = ""
dataset_base64 = ""
target_column = ""
test_size = 0.2
regularization = "none"
alpha = 1.0

In [ ]:
import pandas as pd
import numpy as np
import json, io, base64, warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

result = {}
try:
    # ── Load data ──────────────────────────────────────────────────────────────
    if dataset_url:
        ext = dataset_url.split('.')[-1].lower()
        df = (pd.read_excel(dataset_url) if ext in ['xlsx', 'xls']
              else pd.read_json(dataset_url) if ext == 'json'
              else pd.read_csv(dataset_url))
    elif dataset_base64:
        raw = base64.b64decode(dataset_base64)
        try:
            df = pd.read_csv(io.BytesIO(raw))
        except Exception:
            df = pd.read_excel(io.BytesIO(raw))
    else:
        from sklearn.datasets import load_diabetes
        diab = load_diabetes()
        df = pd.DataFrame(diab.data, columns=diab.feature_names)
        df['target'] = diab.target
        if not target_column:
            target_column = 'target'

    tgt = target_column if target_column else df.columns[-1]
    if tgt not in df.columns:
        raise ValueError(f"Target column '{tgt}' not found. Available: {list(df.columns)}")

    # ── Preprocessing ──────────────────────────────────────────────────────────
    df = df.dropna(subset=[tgt]).copy()
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'unknown')
    for col in df.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    X = df.drop(columns=[tgt])
    y = df[tgt]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=float(test_size), random_state=42)

    # ── Train ──────────────────────────────────────────────────────────────────
    reg = str(regularization).lower() if regularization else 'none'
    if reg == 'ridge':
        model = Ridge(alpha=float(alpha))
    elif reg == 'lasso':
        model = Lasso(alpha=float(alpha), max_iter=10000)
    else:
        model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = float(mean_squared_error(y_test, y_pred))
    result = {
        'algorithm': 'linear_regression',
        'regularization': reg,
        'metrics': {
            'r2_score': round(float(r2_score(y_test, y_pred)), 4),
            'rmse':     round(float(np.sqrt(mse)), 4),
            'mae':      round(float(mean_absolute_error(y_test, y_pred)), 4),
            'mse':      round(mse, 4),
        }
    }

    coeffs = model.coef_.tolist() if hasattr(model.coef_, 'tolist') else list(model.coef_)
    result['coefficients'] = [
        {'feature': col, 'coefficient': round(float(c), 4)}
        for col, c in zip(X.columns.tolist(), coeffs)
    ]

    n_s = min(50, len(y_test))
    result['predicted_vs_actual'] = [
        {'actual': round(float(a), 4), 'predicted': round(float(p), 4)}
        for a, p in zip(y_test.values[:n_s], y_pred[:n_s])
    ]

except Exception as e:
    import traceback
    result = {'error': str(e), 'traceback': traceback.format_exc()}

with open('/tmp/ml_result.json', 'w') as f:
    json.dump(result, f)
print(json.dumps(result, indent=2))